# Customer Churn Prediction - Data Pipeline & MongoDB Integration

### Data Ingestion, Cleaning, and Prediction Pipeline

This notebook implements the data pipeline stage of the churn prediction project. It focuses on integrating the dataset with MongoDB, performing data ingestion and cleaning, and applying the trained machine learning model to generate predictions on customer data.

The goal of this stage is to simulate a production-oriented workflow where data is stored, retrieved, and processed through a predictive pipeline.

---

## Pipeline Overview

- Connect to MongoDB database  
- Ingest and store customer data  
- Clean and preprocess data  
- Retrieve data for analysis  
- Apply trained churn prediction model  
- Generate predictions on customer records

## 1. Connect to MongoDB


In [2]:
import pymongo
import pandas as pd
from pymongo import MongoClient

In [3]:
client = MongoClient('mongodb://localhost:27017/')

In [4]:
client.list_database_names()

['Tech', 'admin', 'atest', 'config', 'local']

## 2. Select the Database and Collection

We choose the **Tech** database and verify that the **Customers** collection exists.



In [5]:
db = client['Tech']

In [6]:
db.list_collection_names()

['Customers']

In [7]:
db.Customers.find({})

## 3. Inspect Collection Fields

To ensure data consistency, we extract all unique field names that appear across documents in the Customers collection.



In [8]:
all_fields = set()

for doc in db.Customers.find():
    all_fields.update(doc.keys())

print(all_fields)

{'SeniorCitizen', 'customerID', 'PaymentMethod', 'Churn', 'Contract', '_id', 'MultipleLines', 'gender', 'Partner', 'TotalCharges', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'PhoneService', 'tenure', 'PaperlessBilling', 'MonthlyCharges', 'StreamingTV', 'TechSupport', 'InternetService', 'Dependents', 'StreamingMovies'}


## 4. Filter Documents Containing All Required Fields

We build a MongoDB query that returns only documents where **every field exists**, ensuring a complete dataset for analysis.

In [9]:
query_all_fields = {field: {"$exists": True} for field in all_fields}
query_all_fields

{'SeniorCitizen': {'$exists': True},
 'customerID': {'$exists': True},
 'PaymentMethod': {'$exists': True},
 'Churn': {'$exists': True},
 'Contract': {'$exists': True},
 '_id': {'$exists': True},
 'MultipleLines': {'$exists': True},
 'gender': {'$exists': True},
 'Partner': {'$exists': True},
 'TotalCharges': {'$exists': True},
 'OnlineSecurity': {'$exists': True},
 'OnlineBackup': {'$exists': True},
 'DeviceProtection': {'$exists': True},
 'PhoneService': {'$exists': True},
 'tenure': {'$exists': True},
 'PaperlessBilling': {'$exists': True},
 'MonthlyCharges': {'$exists': True},
 'StreamingTV': {'$exists': True},
 'TechSupport': {'$exists': True},
 'InternetService': {'$exists': True},
 'Dependents': {'$exists': True},
 'StreamingMovies': {'$exists': True}}

## 5. Retrieve Documents and Build the Initial DataFrame

We query MongoDB to extract documents that contain all required fields, remove the `_id` field, and convert the results into a Pandas DataFrame for further processing.



In [10]:
docs_with_all_fields = list(db.Customers.find(query_all_fields))
docs_with_all_fields

[{'_id': ObjectId('69b8078857b6ec47a2d0c8e1'),
  'customerID': '7590-VHVEG',
  'gender': 'Female',
  'SeniorCitizen': 0,
  'Partner': 'Yes',
  'Dependents': 'No',
  'tenure': 1,
  'PhoneService': 'No',
  'MultipleLines': 'No phone service',
  'InternetService': 'DSL',
  'OnlineSecurity': 'No',
  'OnlineBackup': 'Yes',
  'DeviceProtection': 'No',
  'TechSupport': 'No',
  'StreamingTV': 'No',
  'StreamingMovies': 'No',
  'Contract': 'Month-to-month',
  'PaperlessBilling': 'Yes',
  'PaymentMethod': 'Electronic check',
  'MonthlyCharges': 29.85,
  'TotalCharges': 29.85,
  'Churn': 'No'},
 {'_id': ObjectId('69b8078857b6ec47a2d0c8e2'),
  'customerID': '5575-GNVDE',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'No',
  'Dependents': 'No',
  'tenure': 34,
  'PhoneService': 'Yes',
  'MultipleLines': 'No',
  'InternetService': 'DSL',
  'OnlineSecurity': 'Yes',
  'OnlineBackup': 'No',
  'DeviceProtection': 'Yes',
  'TechSupport': 'No',
  'StreamingTV': 'No',
  'StreamingMovies': 'No',
  

### 5.1 Remove the `_id` Field


In [11]:
docs_no_id = list(db.Customers.find({}, {"_id": 0}))
docs_no_id[:2]  

[{'customerID': '7590-VHVEG',
  'gender': 'Female',
  'SeniorCitizen': 0,
  'Partner': 'Yes',
  'Dependents': 'No',
  'tenure': 1,
  'PhoneService': 'No',
  'MultipleLines': 'No phone service',
  'InternetService': 'DSL',
  'OnlineSecurity': 'No',
  'OnlineBackup': 'Yes',
  'DeviceProtection': 'No',
  'TechSupport': 'No',
  'StreamingTV': 'No',
  'StreamingMovies': 'No',
  'Contract': 'Month-to-month',
  'PaperlessBilling': 'Yes',
  'PaymentMethod': 'Electronic check',
  'MonthlyCharges': 29.85,
  'TotalCharges': 29.85,
  'Churn': 'No'},
 {'customerID': '5575-GNVDE',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'No',
  'Dependents': 'No',
  'tenure': 34,
  'PhoneService': 'Yes',
  'MultipleLines': 'No',
  'InternetService': 'DSL',
  'OnlineSecurity': 'Yes',
  'OnlineBackup': 'No',
  'DeviceProtection': 'Yes',
  'TechSupport': 'No',
  'StreamingTV': 'No',
  'StreamingMovies': 'No',
  'Contract': 'One year',
  'PaperlessBilling': 'No',
  'PaymentMethod': 'Mailed check',
  'Mont

### 5.2 Create a Clean Dataset

We combine both conditions: documents must contain all fields and exclude the `_id` field. The result is converted into a DataFrame.



In [12]:
clean_docs = list(db.Customers.find(query_all_fields, {"_id": 0}))
df = pd.DataFrame(clean_docs)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 6. Inspect DataFrame Structure


In [13]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [14]:
df.shape

(7043, 21)

In [15]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges         object
Churn                   str
dtype: object

## 7. Schema Cleaning and Adjustment

Using insights from MongoDB Compass, we identify non-scalar or inconsistent fields. The `TotalCharges` column contains non-numeric values, so we convert it to numeric and drop rows where conversion fails.


In [16]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"])

In [17]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

## 8. Reapply Preprocessing Steps from Part A

To ensure consistency with the original machine learning pipeline, we drop the same columns removed in Part A before encoding and modeling.


In [18]:
df = df.drop(columns=['customerID', 'gender', 'PhoneService', 'MultipleLines'])

In [19]:
df.head(3)

,SeniorCitizen,Partner,Dependents,tenure,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Yes,No,1,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,0,No,No,34,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,0,No,No,2,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [20]:
df.shape

(7032, 17)

## 9. One‑Hot Encode Categorical Features

We apply one‑hot encoding using the same settings as in Part A to ensure the model receives identical feature representations.

In [21]:
encoded = pd.get_dummies(df, drop_first=True)

## 10. Select Final Feature Set
We select the exact same features used by the best model in Part A to maintain consistency between training and prediction.


In [22]:
selected_features = [
    'tenure',
    'Contract_One year',
    'Contract_Two year',
    'InternetService_Fiber optic',
    'PaymentMethod_Electronic check',
    'OnlineSecurity_Yes',
    'TechSupport_Yes',
    'SeniorCitizen',
    'Partner_Yes',
    'Dependents_Yes'
]

In [23]:
X = encoded[selected_features]
X.columns

Index(['tenure', 'Contract_One year', 'Contract_Two year',
       'InternetService_Fiber optic', 'PaymentMethod_Electronic check',
       'OnlineSecurity_Yes', 'TechSupport_Yes', 'SeniorCitizen', 'Partner_Yes',
       'Dependents_Yes'],
      dtype='str')

In [24]:
encoded.columns.tolist()

['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'Partner_Yes',
 'Dependents_Yes',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_One year',
 'Contract_Two year',
 'PaperlessBilling_Yes',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check',
 'Churn_Yes']

## 11. Prepare the Target Variable

The target column `Churn_Yes` is extracted and converted from boolean to integer (0 = No, 1 = Yes).


In [25]:
y = encoded['Churn_Yes']
y

0       False
1       False
2        True
3       False
4        True
        ...  
7038    False
7039    False
7040    False
7041     True
7042    False
Name: Churn_Yes, Length: 7032, dtype: bool

In [26]:
y = encoded['Churn_Yes'].astype(int)

In [27]:
y

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: Churn_Yes, Length: 7032, dtype: int64

## 12. Load and Train the Best Model
We recreate the best-performing model from Part A (Random Forest) and train it on the processed dataset.


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [29]:
from sklearn.ensemble import RandomForestClassifier

best_rf = RandomForestClassifier(
    max_depth=10,
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=4,
    random_state=42
)

best_rf.fit(X_train[selected_features], y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

## 13. Generate Churn Predictions

The trained model is used to generate churn predictions for all encoded rows.



In [30]:
encoded["PredictedChurn"] = best_rf.predict(encoded[selected_features])
encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Partner_Yes,Dependents_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes,PredictedChurn
0,0,1,29.85,29.85,True,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,1
1,0,34,56.95,1889.50,False,False,False,False,False,True,...,False,False,True,False,False,False,False,True,False,0
2,0,2,53.85,108.15,False,False,False,False,False,True,...,False,False,False,False,True,False,False,True,True,0
3,0,45,42.30,1840.75,False,False,False,False,False,True,...,False,False,True,False,False,False,False,False,False,0
4,0,2,70.70,151.65,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,True,1


In [31]:
df["PredictedChurn"] = encoded["PredictedChurn"]

In [32]:
df.head()

,SeniorCitizen,Partner,Dependents,tenure,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,PredictedChurn
0,0,Yes,No,1,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1
1,0,No,No,34,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,0,No,No,2,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0
3,0,No,No,45,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,0,No,No,2,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [33]:
df.shape

(7032, 18)

## 14. Add Predictions to the Original Dataset

We merge the predicted churn values back into the original dataset (with all original fields preserved), creating the final output required for submission.


In [34]:
df_original = pd.DataFrame(clean_docs)

In [35]:
df_original["PredictedChurn"] = encoded["PredictedChurn"]

In [36]:
df_original.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,PredictedChurn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1.0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.5,No,0.0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0.0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0.0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1.0


In [37]:
df_original.shape

(7043, 22)

## 15. Export Final Dataset

The final dataset is saved as `churn.csv`, matching the structure of the original file with an additional `PredictedChurn` column.


In [38]:
df_original.to_csv("churn.csv", index=False)

## 16. Conclusion

In this notebook, we successfully:
- Retrieved customer data from MongoDB
- Cleaned and validated the schema
- Reapplied the preprocessing pipeline from Part A
- Loaded the best-performing churn prediction model
- Generated churn predictions for all customers
- Restored the original dataset structure
- Exported the final `churn.csv` file with the new `PredictedChurn` column
